In [ ]:
%matplotlib inline
import pandas as pd 
import numpy as np
from pandas import DataFrame 
import matplotlib.pyplot as plt

df = pd.read_csv('mnist_train.csv')
print(df.shape)
print(df.head())

In [ ]:
df.head(1)

In [ ]:
df.iloc[1]

In [ ]:
df.iloc[1].to_numpy()


In [ ]:
img1 = df.iloc[1, 1:785].to_numpy()
img1.reshape(28, 28)

In [ ]:
plt.imshow(img1.reshape(28, 28), cmap='gray')
plt.show()

In [ ]:
Y = df.iloc[:, 0].to_numpy()
X = df.iloc[:, 1:].to_numpy()

outX = np.divide(X, 255)
print(outX.shape)
print(Y.shape)
np.unique(Y)



In [ ]:
trainX = outX[:48000, :]
testX = outX[48000:, :]
trainY = Y[:48000]
testY = Y[48000:]
print(trainX.shape)
print(testX.shape)
print(trainY.shape)
print(testY.shape)

In [ ]:
c = 0.01 
W1 = np.random.randn(128, 784) * c
b1 = np.random.randn(128, 1)*c
W2 = np.random.randn(10, 128)*c
b2 = np.random.randn(10, 1)*c

In [ ]:
print(trainY[:10])

In [12]:
encodedY = np.zeros((trainY.size, trainY.max()+1))
encodedY[np.arange(trainY.size), trainY] = 1



In [ ]:
print(trainY[:10])

In [14]:
trainX.shape
trainX = trainX.T


In [71]:
def Forward_pass(X, W1, b1, W2, b2):
    Z1 = np.dot(W1, X) + b1
    A1 = np.maximum(0, Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = np.exp(Z2)/np.sum(np.exp(Z2), axis=0, keepdims=True)
    return A1, A2, Z1
    A1, A2, Z1 = Forward_pass(X, W1, b1, W2, b2)



In [75]:
A1, A2, Z1 = Forward_pass(trainX, W1, b1, W2, b2)


In [ ]:
print(A2[:, 0])
print(np.sum(A2[:, 0]))

In [ ]:
A2.shape
F_R = np.empty([1, 48000], dtype=int)
print(np.argmax(A2, axis=0, keepdims=True, out=F_R))

F_R.shape
A2 = A2.T


In [ ]:
print(A2.shape)
print(encodedY.shape)

In [88]:
def Backward_pass(A1, A2, Z1, X, Y, W2):
    dz2 = A2 - Y
    dw2 = np.dot(dz2, A1.T)/48000
    db2 = np.sum(dz2, axis=1, keepdims=True) / 48000
    dz1 = np.dot(W2.T, dz2) * (Z1 > 0).astype(float)
    dw1 = np.dot(dz1, X.T)/48000
    db1 = np.sum(dz1, axis=1, keepdims=True)/48000
    return dw1, db1, dw2, db2

def update_weights(W1, b1, W2, b2, dw1, db1, dw2, db2, lr ):
    W1 = W1 - lr * dw1
    b1 = b1 - lr * db1
    W2 = W2 - lr * dw2
    b2 = b2 - lr * db2

    return W1,  b1, W2, b2

def train(X, Y, W1, b1, W2, b2, lr ):
    for i in range(500):
        A1, A2, Z1 = Forward_pass( X, W1, b1, W2, b2)
        dw1, db1, dw2, db2 = Backward_pass(A1, A2, Z1, X, Y, W2)
        W1, b1, W2, b2 = update_weights(W1, b1, W2, b2, dw1, db1, dw2, db2, lr)
        loss = -1/48000 * np.sum(Y * np.log(A2))
        if i % 50 == 0:
            print(loss)
    return W1, b1, W2, b2  

In [105]:
W1, b1, W2, b2 = train(trainX, encodedY.T, W1, b1, W2, b2, 0.1)
A1, A2, Z1 = Forward_pass(testX.T, W1, b1, W2, b2)
predictions = np.argmax(A2, axis=0)
accuracy = np.sum(predictions==testY)/ testY.size * 100
print("Accuracy: ", accuracy)

0.34111395198429983
0.3311846553021494
0.3225073886574864
0.3147408387566033
0.3076907327656635
0.3011833583106406
0.29508944250739494
0.28931945129674824
0.28379356954893803
0.2784745146345964
Accuracy:  92.49166666666667


In [ ]:
print("Accuracy: ", accuracy)
def visualise():
    for i in range(20):
        img = testX[i, :].reshape(28, 28)
        plt.figure()
        plt.imshow(img, cmap='summer')
        plt.title(f"Predicted: {predictions[i]} Real: {testY[i]}")
        plt.show()

In [ ]:
visualise()